# Ling 3.0 Tiny: LoRA + agentic GSPO on AppWorld
This notebook runs a deliberately tiny, real before/after experiment. AppWorld database-state evaluators supply rewards; held-out `dev` task IDs are never used for training. The smoke result validates the pipeline, not the hypothesis.

## 1. Runtime and GPU check

In [ ]:
import json, os, platform, subprocess, sys
import torch
assert platform.system() == 'Linux', 'Use a Google Colab Linux runtime.'
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > NVIDIA GPU.'
props = torch.cuda.get_device_properties(0)
runtime = {'gpu': props.name, 'vram_gib': round(props.total_memory/2**30, 1), 'cuda': torch.version.cuda, 'torch': torch.__version__}
print(json.dumps(runtime, indent=2))
assert runtime['vram_gib'] >= 30, 'Ling Tiny + AReno training is not expected to fit reliably below 30 GiB. Choose an A100 runtime.'

## 2. Clone and install this repository
Set `REPO_URL` to the GitHub repository containing this notebook. A private repository requires a suitable Git credential or token in the URL. No model token is printed.

In [ ]:
from pathlib import Path
REPO_URL = os.environ.get('LING_RL_REPO_URL', 'https://github.com/YOUR_USER/YOUR_REPO.git')
assert 'YOUR_USER' not in REPO_URL, 'Edit REPO_URL in this cell before Run all.'
PROJECT = Path('/content/ling-agent-rl-poc')
if not PROJECT.exists(): subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)

## 3. Install AReno and AppWorld
AReno builds its CUDA extension from the audited source revision. If Colab explicitly requests a restart after installation, restart once and continue at section 4.

In [ ]:
ARENO_REV = '48d07c54051c41bf36218f99bce1c3697e9ba63c'
APPWORLD_REV = '42b5bcf3cd334fee33f0c37c02070a9f5807add5'
areno_src = Path('/content/AReno')
if not areno_src.exists(): subprocess.run(['git', 'clone', 'https://github.com/inclusionAI/AReno.git', str(areno_src)], check=True)
subprocess.run(['git', '-C', str(areno_src), 'checkout', ARENO_REV], check=True)
subprocess.run(['bash', str(areno_src/'scripts/install.sh')], cwd=areno_src, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', f'git+https://github.com/StonyBrookNLP/appworld.git@{APPWORLD_REV}'], check=True)
subprocess.run(['appworld', 'install'], check=True)
subprocess.run(['appworld', 'download', 'data'], check=True)

## 4. Validate imports, credentials, and experiment split

In [ ]:
os.chdir(PROJECT)
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token: os.environ['HF_TOKEN'] = token
except Exception: pass
import areno, appworld, ling_agent_rl
subprocess.run([sys.executable, '-m', 'pytest'], check=True)  # includes a real AppWorld CPU interaction now that data exists
from ling_agent_rl.config import load_config
from ling_agent_rl.train import prepare_dataset
CONFIG_PATH = 'configs/smoke.yaml'  # change to configs/experiment.yaml only after smoke succeeds
config = load_config(CONFIG_PATH)
dataset_path, train_ids, eval_ids = prepare_dataset(config)
assert set(train_ids).isdisjoint(eval_ids)
print({'train_tasks': train_ids, 'held_out_tasks': eval_ids})

## 5. Start baseline Ling and run one real trajectory

In [ ]:
import time, urllib.request
def start_server(adapter=None, port=8000):
    cmd = ['areno','serve','--model-path',config.model,'--model-hub','hf','--world-size','1','--tp-size','1','--port',str(port)]
    if adapter: cmd += ['--lora-adapter-path', str(adapter)]
    proc = subprocess.Popen(cmd)
    for _ in range(180):
        if proc.poll() is not None: raise RuntimeError(f'AReno server exited with {proc.returncode}')
        try:
            urllib.request.urlopen(f'http://127.0.0.1:{port}/v1/models', timeout=2); return proc
        except Exception: time.sleep(2)
    proc.terminate(); raise TimeoutError('AReno server did not become ready')
baseline_server = start_server()

In [ ]:
import asyncio
from openai import AsyncOpenAI
from ling_agent_rl.agent import run_episode
async def one_real_episode():
    client = AsyncOpenAI(base_url='http://127.0.0.1:8000/v1', api_key='unused')
    try:
        return (await run_episode(client=client, task_id=eval_ids[0], model='policy', rollout_config=config.rollout, experiment_name='ling_rl_preflight'))[0]
    finally: await client.close()
preflight = asyncio.run(one_real_episode())
print(json.dumps(preflight.to_dict(), indent=2, default=str))
assert preflight.tool_calls > 0, 'Ling did not make an AppWorld tool call; inspect the printed response/tool format.'

## 6. Held-out baseline evaluation

In [ ]:
from ling_agent_rl.evaluate import run_evaluation
baseline = run_evaluation(config, eval_ids, base_url='http://127.0.0.1:8000', label='baseline')
print(json.dumps(baseline, indent=2))
baseline_server.terminate(); baseline_server.wait(timeout=30)

## 7. One real native-LoRA GSPO update and adapter save
This invokes AReno's native Bailing-MoE V3 LoRA and agentic rollout hook. It does not silently fall back to full-parameter training.

In [ ]:
from ling_agent_rl.train import train, verify_adapter
train(config)
adapter = verify_adapter(Path(config.artifact_dir)/'adapter')
print('Validated adapter:', adapter)

## 8. Reload the adapter and evaluate the trained policy

In [ ]:
trained_server = start_server(adapter=adapter)
trained = run_evaluation(config, eval_ids, base_url='http://127.0.0.1:8000', label='trained')
trained_server.terminate(); trained_server.wait(timeout=30)
print(json.dumps(trained, indent=2))

## 9. Before/after comparison

In [ ]:
from ling_agent_rl.evaluate import compare
comparison = compare(baseline, trained)
comparison_path = Path(config.artifact_dir)/'comparison.json'
comparison_path.write_text(json.dumps(comparison, indent=2))
print(json.dumps(comparison, indent=2))
print('Smoke deltas validate wiring only; use the larger multi-seed experiment for evidence.')

## 10. Optional Google Drive artifact copy

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive/ling-agent-rl-artifacts')/Path(config.artifact_dir).name
    shutil.copytree(config.artifact_dir, destination, dirs_exist_ok=True)
    print(destination)